# Iris 資料集機器學習期末實作

## 1. 資料載入與初步處理

In [ ]:
from sklearn.datasets import load_iris
import pandas as pd
iris = load_iris()
df = pd.DataFrame(iris.data, columns=iris.feature_names)
df['target'] = iris.target
df.head()

## 2. 特徵視覺化

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt
sns.pairplot(df, hue='target')
plt.show()

## 3. 資料標準化

In [ ]:
from sklearn.preprocessing import StandardScaler
scaler = StandardScaler()
X_scaled = scaler.fit_transform(iris.data)

## 4. 訓練/測試集切分

In [ ]:
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(X_scaled, iris.target, test_size=0.2, random_state=42)

## 5. 模型訓練與比較

In [ ]:
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score, confusion_matrix

knn = KNeighborsClassifier(n_neighbors=3)
knn.fit(X_train, y_train)
knn_preds = knn.predict(X_test)
print("KNN Accuracy:", accuracy_score(y_test, knn_preds))

svm = SVC(kernel='linear')
svm.fit(X_train, y_train)
svm_preds = svm.predict(X_test)
print("SVM Accuracy:", accuracy_score(y_test, svm_preds))

## 6. 交叉驗證

In [ ]:
from sklearn.model_selection import cross_val_score
print("KNN CV Accuracy:", cross_val_score(knn, X_scaled, iris.target, cv=5).mean())
print("SVM CV Accuracy:", cross_val_score(svm, X_scaled, iris.target, cv=5).mean())

## 7. 超參數調整

In [ ]:
from sklearn.model_selection import GridSearchCV
param_grid = {'n_neighbors': list(range(1, 11))}
grid_knn = GridSearchCV(KNeighborsClassifier(), param_grid, cv=5)
grid_knn.fit(X_scaled, iris.target)
print("Best KNN Params:", grid_knn.best_params_)

param_grid = {'C': [0.1, 1, 10], 'kernel': ['linear', 'rbf']}
grid_svm = GridSearchCV(SVC(), param_grid, cv=5)
grid_svm.fit(X_scaled, iris.target)
print("Best SVM Params:", grid_svm.best_params_)

## 8. SVM 分類邊界視覺化

In [ ]:
from sklearn.decomposition import PCA
import numpy as np
pca = PCA(n_components=2)
X_pca = pca.fit_transform(X_scaled)
svm_pca = SVC(kernel='linear')
svm_pca.fit(X_pca, iris.target)

x_min, x_max = X_pca[:, 0].min() - 1, X_pca[:, 0].max() + 1
y_min, y_max = X_pca[:, 1].min() - 1, X_pca[:, 1].max() + 1
xx, yy = np.meshgrid(np.linspace(x_min, x_max, 300),
                     np.linspace(y_min, y_max, 300))
Z = svm_pca.predict(np.c_[xx.ravel(), yy.ravel()])
Z = Z.reshape(xx.shape)

plt.contourf(xx, yy, Z, alpha=0.3)
plt.scatter(X_pca[:, 0], X_pca[:, 1], c=iris.target)
plt.title("SVM Decision Boundary after PCA")
plt.show()